<div style="background:linear-gradient(135deg,#0b0f1e,#1a2550,#0f162f);color:#e6e9ff;border-radius:16px;padding:28px 28px 20px;border:1px solid #24335c;box-shadow:0 0 20px rgba(0,0,0,0.4);font-family:'Segoe UI',sans-serif;">

<h1 style="font-size:28px;margin-bottom:6px;">🎧 SMSAT Data Augmentation with <span style="color:#7aa2ff;">WaveGAN</span> + Quality Evaluation</h1>
<p style="margin-top:0;color:#aab5d9;font-size:14px;">Kaggle notebook · single-file pipeline · trains & evaluates across all 3 classes</p>

<p style="margin:6px 0 14px;">
  <span style="background:#162050;padding:6px 10px;border-radius:999px;margin-right:6px;font-size:12px;">🎵 Audio GAN (WaveGAN, 1-sec chunks)</span>
  <span style="background:#162050;padding:6px 10px;border-radius:999px;margin-right:6px;font-size:12px;">🧠 Quality: FAD (YAMNet), MFCC, Classifier</span>
  <span style="background:#162050;padding:6px 10px;border-radius:999px;margin-right:6px;font-size:12px;">⚙️ End-to-End: Train → Generate → Evaluate</span>
  <span style="background:#162050;padding:6px 10px;border-radius:999px;font-size:12px;">📈 XLSX logs & summaries</span>
</p>

<hr style="border:none;border-top:1px solid #2c3a6f;margin:14px 0;">

<h2 style="color:#9bb3ff;">📂 Dataset layout (expected)</h2>
<pre style="background:#0b132b;padding:12px 16px;border-radius:12px;border:1px solid #2b3661;color:#dfe4ff;">
/kaggle/input/qmsat-dataset/ATS-data/
├── Music/
├── Normal(Silence)/
└── SpiritualMeditation/
</pre>
<p style="font-size:13px;color:#9ba6d9;">Place class-specific WAVs inside each folder.</p>

<div style="background:#111b3f;border:1px solid #283667;padding:14px 18px;border-radius:12px;margin-top:14px;">
  <h3 style="color:#7aa2ff;margin-top:0;">📤 Outputs</h3>
  <ul style="margin:0;padding-left:18px;font-size:13px;">
    <li>500 × 60-sec generated WAVs per class → <code>/kaggle/working/output/</code></li>
    <li>Training logs → <code>/kaggle/working/output/{CLS}_training_metrics.xlsx</code></li>
    <li>Quality eval summary → <code>/kaggle/working/output/quality_eval_summary.xlsx</code></li>
  </ul>
</div>

<div style="background:#111b3f;border:1px solid #283667;padding:14px 18px;border-radius:12px;margin-top:14px;">
  <h3 style="color:#79d29d;margin-top:0;">📊 Core Metrics</h3>
  <ul style="margin:0;padding-left:18px;font-size:13px;">
    <li>🎯 Frechet Audio Distance (proxy via YAMNet embeddings)</li>
    <li>🎵 MFCC distribution similarity (cosine)</li>
    <li>🧩 Classifier believability (1D-CNN accuracy on generated)</li>
  </ul>
</div>

<div style="background:#111b3f;border:1px solid #283667;padding:14px 18px;border-radius:12px;margin-top:14px;">
  <h3 style="color:#ffd285;margin-top:0;">📝 Notes</h3>
  <ul style="margin:0;padding-left:18px;font-size:13px;">
    <li>Generate 1-sec WaveGAN audio, stitch 60 chunks → single 60-sec sample.</li>
    <li>For demo runtime, reduce <code>EPOCHS</code>; increase for higher fidelity.</li>
    <li>Metric scripts run per class and aggregate into one Excel summary.</li>
  </ul>
</div>

<hr style="border:none;border-top:1px solid #2c3a6f;margin:20px 0 10px;">
<p style="font-size:13px;color:#aab5d9;">Quick links:</p>
<p>
  <a href="#setup" style="color:#7aa2ff;text-decoration:none;padding:5px 10px;border:1px solid #2f4479;border-radius:999px;font-size:12px;">1️⃣ Setup</a>
  <a href="#data-loading" style="color:#7aa2ff;text-decoration:none;padding:5px 10px;border:1px solid #2f4479;border-radius:999px;font-size:12px;">2️⃣ Data Loading</a>
  <a href="#train-wavegan" style="color:#7aa2ff;text-decoration:none;padding:5px 10px;border:1px solid #2f4479;border-radius:999px;font-size:12px;">3️⃣ Train WaveGAN</a>
  <a href="#generate" style="color:#7aa2ff;text-decoration:none;padding:5px 10px;border:1px solid #2f4479;border-radius:999px;font-size:12px;">4️⃣ Generate</a>
  <a href="#quality" style="color:#7aa2ff;text-decoration:none;padding:5px 10px;border:1px solid #2f4479;border-radius:999px;font-size:12px;">5️⃣ Quality Eval</a>
  <a href="#export" style="color:#7aa2ff;text-decoration:none;padding:5px 10px;border:1px solid #2f4479;border-radius:999px;font-size:12px;">6️⃣ Export Logs</a>
</p>

</div>


In [1]:
# Optional: installs (comment out if preinstalled on Kaggle image)
!pip -q install tensorflow_hub openpyxl

In [2]:
# ===============================
# SMSAT WaveGAN - Music Class Only (3000 epochs, 200 samples)
# Full Notebook with Evaluation
# ===============================

import os, time, glob, random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from tqdm import tqdm
import tensorflow_hub as hub

# -------------------- CONFIG --------------------
DATA_DIR = "/kaggle/input/qmsat-dataset/ATS-data"   # <-- Your dataset path
OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_MAP = {"M": "Music"}   # ✅ Only Music
CLASSES = ["M"]

SAMPLE_RATE = 16000
CHUNK_SECONDS = 1
FILE_SECONDS = 60
Z_DIM = 128
BATCH_SIZE = 32
EPOCHS = 3000
LOG_INTERVAL = 100

# -------------------- IO UTILS --------------------
def list_wavs_for_class(cls_code):
    folder = os.path.join(DATA_DIR, CLASS_MAP[cls_code])
    return sorted([p for p in glob.glob(os.path.join(folder, "*.wav"))])

def chunkify_1s(y, sr=SAMPLE_RATE, chunk_seconds=CHUNK_SECONDS):
    step = sr * chunk_seconds
    chunks = []
    for i in range(0, len(y), step):
        seg = y[i:i+step]
        if len(seg) < step:
            seg = np.pad(seg, (0, step - len(seg)))
        chunks.append(seg)
    return np.array(chunks)

def load_audio_1s_chunks(cls_code, max_files=None):
    files = list_wavs_for_class(cls_code)
    if max_files: files = files[:max_files]
    data = []
    for f in files:
        y, sr = librosa.load(f, sr=SAMPLE_RATE, mono=True)
        chunks = chunkify_1s(y)
        data.extend(chunks)
    return np.array(data)

def save_wav(data, filename, sr=SAMPLE_RATE):
    data = np.clip(data, -1.0, 1.0)
    sf.write(filename, data, sr)

# -------------------- MODELS --------------------
def build_generator(z_dim=Z_DIM, out_len=SAMPLE_RATE*CHUNK_SECONDS):
    return tf.keras.Sequential([
        layers.Input(shape=(z_dim,)),
        layers.Dense(1000, activation="relu"),
        layers.Reshape((1000,1)),
        layers.Conv1DTranspose(128, 25, strides=2, padding="same", activation="relu"),
        layers.Conv1DTranspose(64, 25, strides=2, padding="same", activation="relu"),
        layers.Conv1DTranspose(32, 25, strides=2, padding="same", activation="relu"),
        layers.Conv1DTranspose(16, 25, strides=2, padding="same", activation="relu"),
        layers.Conv1D(1, 7, padding="same", activation="tanh"),
        layers.Reshape((out_len,))
    ], name="Generator")

def build_discriminator(in_len=SAMPLE_RATE*CHUNK_SECONDS):
    return tf.keras.Sequential([
        layers.Input(shape=(in_len,)),
        layers.Reshape((in_len,1)),
        layers.Conv1D(64, 25, strides=4, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Conv1D(128, 25, strides=4, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Conv1D(256, 25, strides=4, padding="same"),
        layers.LeakyReLU(0.2),
        layers.Flatten(),
        layers.Dense(1)
    ], name="Discriminator")

bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
def d_loss(real_out, fake_out): return bce(tf.ones_like(real_out), real_out) + bce(tf.zeros_like(fake_out), fake_out)
def g_loss(fake_out): return bce(tf.ones_like(fake_out), fake_out)

# -------------------- TRAINING --------------------
def train_wavegan_for_class(cls_code, epochs=3000, target_files=200, checkpoint_interval=500):
    print(f"\n=== Training WaveGAN for class: {cls_code} ===")
    real_chunks = load_audio_1s_chunks(cls_code)
    real_chunks = real_chunks.reshape((-1, SAMPLE_RATE*CHUNK_SECONDS)).astype(np.float32)
    dataset = tf.data.Dataset.from_tensor_slices(real_chunks).shuffle(min(10000, len(real_chunks))).batch(BATCH_SIZE, drop_remainder=True)

    G, D = build_generator(), build_discriminator()
    g_opt = tf.keras.optimizers.Adam(1e-4, 0.5, 0.9)
    d_opt = tf.keras.optimizers.Adam(1e-4, 0.5, 0.9)

    metrics, start_time = [], time.time()

    @tf.function
    def train_step(real_batch):
        noise = tf.random.normal([tf.shape(real_batch)[0], Z_DIM])
        with tf.GradientTape() as gt, tf.GradientTape() as dt:
            fake = G(noise, training=True)
            real_out, fake_out = D(real_batch, training=True), D(fake, training=True)
            gl, dl = g_loss(fake_out), d_loss(real_out, fake_out)
        g_opt.apply_gradients(zip(gt.gradient(gl, G.trainable_variables), G.trainable_variables))
        d_opt.apply_gradients(zip(dt.gradient(dl, D.trainable_variables), D.trainable_variables))
        return gl, dl

    for epoch in range(1, epochs+1):
        g_epoch, d_epoch = [], []
        for real_batch in dataset:
            gl, dl = train_step(real_batch)
            g_epoch.append(gl.numpy()); d_epoch.append(dl.numpy())
        elapsed = time.time() - start_time
        g_mean, d_mean = np.mean(g_epoch), np.mean(d_epoch)
        if epoch % LOG_INTERVAL == 0 or epoch in [1, epochs]:
            print(f"Epoch {epoch:5d} | G: {g_mean:.4f} | D: {d_mean:.4f} | elapsed {elapsed/60:.2f}m")
        metrics.append([epoch, g_mean, d_mean, elapsed])

        if epoch % checkpoint_interval == 0 or epoch == epochs:
            G.save(os.path.join(OUTPUT_DIR, f"{cls_code}_generator_epoch{epoch}.h5"))
            D.save(os.path.join(OUTPUT_DIR, f"{cls_code}_discriminator_epoch{epoch}.h5"))

    # Save metrics
    df = pd.DataFrame(metrics, columns=["epoch", "gen_loss", "disc_loss", "elapsed_s"])
    df.to_excel(os.path.join(OUTPUT_DIR, f"{cls_code}_training_metrics.xlsx"), index=False)

    # Loss curve
    plt.figure(figsize=(8,5))
    plt.plot(df["epoch"], df["gen_loss"], label="Generator Loss", color="blue")
    plt.plot(df["epoch"], df["disc_loss"], label="Discriminator Loss", color="red")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title(f"Training Loss Curve - {cls_code}")
    plt.legend(); plt.grid(True); plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{cls_code}_loss_curve.png"))
    plt.close()

    # Generate samples
    print(f"Generating {target_files} x {FILE_SECONDS}s samples for {cls_code} ...")
    for i in tqdm(range(target_files)):
        chunks = []
        for _ in range(FILE_SECONDS):
            z = tf.random.normal([1, Z_DIM])
            one_sec = G(z, training=False).numpy().reshape(-1)
            chunks.append(one_sec)
        full = np.concatenate(chunks, axis=0)
        save_wav(full, os.path.join(OUTPUT_DIR, f"{cls_code}_gen_{i:04d}.wav"))

    print(f"✅ Done {cls_code} in {(time.time()-start_time)/60:.2f} minutes")
    return df, G

# -------------------- EVALUATION --------------------
def load_yamnet(): return hub.load("https://tfhub.dev/google/yamnet/1")

def yamnet_embeddings(y, model): 
    y = y.astype(np.float32); y = y/np.max(np.abs(y)) if np.max(np.abs(y))>0 else y
    _, emb, _ = model(y); return emb.numpy()

def frechet_distance(mu1, sigma1, mu2, sigma2):
    from scipy.linalg import sqrtm
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean): covmean = covmean.real
    diff = mu1 - mu2
    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2*covmean)

def compute_fad_like(real_files, gen_files, max_files=50):
    model, real_embeds, gen_embeds = load_yamnet(), [], []
    for f in real_files[:max_files]:
        y,_ = librosa.load(f, sr=SAMPLE_RATE, mono=True)
        real_embeds.append(yamnet_embeddings(y, model))
    for f in gen_files[:max_files]:
        y,_ = librosa.load(f, sr=SAMPLE_RATE, mono=True)
        gen_embeds.append(yamnet_embeddings(y, model))
    real_embeds = np.vstack([e.mean(axis=0) for e in real_embeds])
    gen_embeds  = np.vstack([e.mean(axis=0) for e in gen_embeds])
    return frechet_distance(real_embeds.mean(0), np.cov(real_embeds,rowvar=False),
                            gen_embeds.mean(0), np.cov(gen_embeds,rowvar=False))

def mfcc_distribution_similarity(real_files, gen_files, max_files=50):
    def to_vec(f):
        y,_ = librosa.load(f, sr=SAMPLE_RATE, mono=True)
        mf = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=20)
        return np.concatenate([mf.mean(1), mf.std(1)])
    real_vecs = np.array([to_vec(f) for f in real_files[:max_files]])
    gen_vecs  = np.array([to_vec(f) for f in gen_files[:max_files]])
    return float(np.dot(real_vecs.mean(0), gen_vecs.mean(0)) /
                 (np.linalg.norm(real_vecs.mean(0))*np.linalg.norm(gen_vecs.mean(0))+1e-8))

def classifier_believability(gen_dir):
    label_map = {"M":0}
    # Real chunks
    Xr, Yr = [], []
    chunks = load_audio_1s_chunks("M", max_files=10)
    Xr.append(chunks); Yr.append(np.zeros(len(chunks)))
    Xr, Yr = np.concatenate(Xr).astype(np.float32), np.concatenate(Yr)

    # Generated chunks
    files = sorted(glob.glob(os.path.join(gen_dir, "M_gen_*.wav")))[:20]
    Xg, Yg = [], []
    for f in files:
        y,_ = librosa.load(f, sr=SAMPLE_RATE, mono=True)
        chunks = chunkify_1s(y)
        Xg.append(chunks); Yg.append(np.zeros(len(chunks)))
    Xg, Yg = np.concatenate(Xg).astype(np.float32), np.concatenate(Yg)

    # CNN
    model = tf.keras.Sequential([
        layers.Input(shape=(SAMPLE_RATE,)),
        layers.Reshape((SAMPLE_RATE,1)),
        layers.Conv1D(32, 9, activation="relu"), layers.MaxPool1D(4),
        layers.Conv1D(64, 9, activation="relu"), layers.MaxPool1D(4),
        layers.Flatten(), layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(Xr, Yr, epochs=5, batch_size=64, verbose=0)
    _, acc = model.evaluate(Xg, Yg, verbose=0)
    return float(acc)

def evaluate_quality():
    real_files = list_wavs_for_class("M")
    gen_files  = sorted(glob.glob(os.path.join(OUTPUT_DIR, "M_gen_*.wav")))

    fad = compute_fad_like(real_files, gen_files)
    mfcc_sim = mfcc_distribution_similarity(real_files, gen_files)
    clf_acc = classifier_believability(OUTPUT_DIR)

    df = pd.DataFrame([{
        "class":"Music",
        "FAD_like": fad,
        "MFCC_cosine_sim": mfcc_sim,
        "Classifier_believability_acc": clf_acc
    }])
    df.to_excel(os.path.join(OUTPUT_DIR, "quality_eval_music.xlsx"), index=False)

    # Plot metrics bar chart
    plt.figure(figsize=(6,4))
    plt.bar(["FAD_like","MFCC_sim","Classifier_acc"], [fad, mfcc_sim, clf_acc])
    plt.title("Quality Metrics - Music GAN")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,"quality_eval_music.png"))
    plt.close()

    print("\n=== Quality Evaluation ===")
    print(df)

# -------------------- RUN --------------------
df, G = train_wavegan_for_class("M", epochs=EPOCHS, target_files=200)
evaluate_quality()

print("\n✅ Finished Music class training (3000 epochs) with 200 samples + evaluation.")
print("Files saved to:", OUTPUT_DIR)


2025-10-04 14:24:25.524917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759587865.762115      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759587865.826438      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered



=== Training WaveGAN for class: M ===


I0000 00:00:1759587895.374608      36 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1759587901.020639     108 cuda_dnn.cc:529] Loaded cuDNN version 90300


Epoch     1 | G: 0.6905 | D: 1.2706 | elapsed 0.14m
Epoch   100 | G: 1.1339 | D: 0.9198 | elapsed 2.55m
Epoch   200 | G: 1.5618 | D: 0.9392 | elapsed 4.96m
Epoch   300 | G: 1.2434 | D: 1.2360 | elapsed 7.37m
Epoch   400 | G: 1.4192 | D: 0.8654 | elapsed 9.78m
Epoch   500 | G: 1.6395 | D: 0.6917 | elapsed 12.20m
Epoch   600 | G: 1.6395 | D: 0.7509 | elapsed 14.61m
Epoch   700 | G: 1.7417 | D: 0.6853 | elapsed 17.03m
Epoch   800 | G: 1.5396 | D: 0.6360 | elapsed 19.44m
Epoch   900 | G: 1.7392 | D: 0.7858 | elapsed 21.85m
Epoch  1000 | G: 1.7139 | D: 0.6825 | elapsed 24.25m
Epoch  1100 | G: 2.0860 | D: 1.1140 | elapsed 26.66m
Epoch  1200 | G: 1.8039 | D: 0.6305 | elapsed 29.08m
Epoch  1300 | G: 1.9481 | D: 0.5728 | elapsed 31.50m
Epoch  1400 | G: 2.6266 | D: 0.9438 | elapsed 33.92m
Epoch  1500 | G: 2.0340 | D: 0.6065 | elapsed 36.34m
Epoch  1600 | G: 2.0408 | D: 0.4913 | elapsed 38.76m
Epoch  1700 | G: 2.1268 | D: 0.4901 | elapsed 41.17m
Epoch  1800 | G: 2.2270 | D: 0.3729 | elapsed 43.57

100%|██████████| 200/200 [02:30<00:00,  1.33it/s]


✅ Done M in 74.87 minutes


I0000 00:00:1759592412.778762     107 service.cc:148] XLA service 0x7e191802d5d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1759592412.781874     107 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1759592416.612263     107 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.



=== Quality Evaluation ===
   class  FAD_like  MFCC_cosine_sim  Classifier_believability_acc
0  Music  41.97437         0.942198                           1.0

✅ Finished Music class training (3000 epochs) with 200 samples + evaluation.
Files saved to: /kaggle/working/output


In [3]:
import shutil, os

# Path where Kaggle saves outputs
OUT_DIR = "/kaggle/working"

# Zip file name
zip_path = "/kaggle/working/output_results.zip"

# Remove old zip if exists
if os.path.exists(zip_path):
    os.remove(zip_path)

# Create new zip (recursively includes all files in working dir)
shutil.make_archive(zip_path.replace(".zip",""), 'zip', OUT_DIR)

print(f"✅ Zipped all outputs to {zip_path}")


✅ Zipped all outputs to /kaggle/working/output_results.zip
